In [ ]:
from functions.preprocessing.data_processing import TractionDataProcessor
from functions.CSVLoader import CSVLoader


In [ ]:
min_height = 20
threshold_on_T = 2500
min_seconds_between_peaks = 30
window_length = 30       # seconds
pre_peak_time = 0.0     # seconds
features_per_signal = 25

signal_names = [
    "V_kite_0_mag",
    "kite_0_latitude",
    "kite_0_longitude",
    "kite_0_altitude",
    "kite_0_roll",
    "kite_0_pitch",
    "kite_0_yaw",
    "V_kite_1_mag",
    "kite_1_latitude",
    "kite_1_longitude",
    "kite_1_altitude",
    "kite_1_roll",
    "kite_1_pitch",
    "kite_1_yaw",
    "ground_tether_reelout_speed",
    "ground_tether_length",
    "ground_tether_force",
    "kite_pos_east",
    "kite_pos_north",
    "kite_height",
    "kite_elevation",
    "kite_azimuth",
    "kite_distance",
    "kite_heading",
    "kite_course",
    "kite_actual_steering",
    "kite_actual_depower",
    "ground_mech_power",
    "kite_turn_rate",
]

In [ ]:
folder = "../data/dataset_1/cleaned/test_data"
folder = CSVLoader(folder)
data = folder.load_all()

In [ ]:
one_data = data[0]

In [ ]:
processor = TractionDataProcessor(
        min_height,
        threshold_on_T,
        min_seconds_between_peaks,
        window_length,
        pre_peak_time,
        plot=True
    )
results = processor.process(one_data)

In [ ]:
results.keys()

In [ ]:
from functions.preprocessing.plots import plot_overlapping_frames_for_signal
plot_overlapping_frames_for_signal(
        signal_name="ground_tether_force",
        raw_df=results["filtered_data"],  # Use filtered data to match frame indices
        processing_output=results,
        window_length=processor.window_length,
        label_value=1,
        threshold=results.get("threshold_on_T", threshold_on_T)  # Use threshold from results if available
    )

In [ ]:
from functions.preprocessing.plots import plot_overlapping_frames_for_signal
plot_overlapping_frames_for_signal(
        signal_name="ground_tether_force",
        raw_df=results["filtered_data"],  # Use filtered data to match frame indices
        processing_output=results,
        window_length=processor.window_length,
        label_value=0,
        threshold=results.get("threshold_on_T", threshold_on_T)  # Use threshold from results if available
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


def plot_overlapping_frames_signal_vs_signal(
    x_signal_name: str,
    y_signal_name: str,
    raw_df: pd.DataFrame,
    processing_output: dict,
    window_length: float,
    label_value: int = 1,
    pre_peak_time: float = None,
    figsize=(6, 6),
    title_suffix: str = "",
):
    """
    Plot overlapped frames where frame_labels == label_value,
    plotting y_signal_name vs x_signal_name.

    Optionally extends each frame with a continuation segment
    (e.g. up to the corresponding peak).

    Parameters
    ----------
    x_signal_name : str
        Column name used for x-axis.
    y_signal_name : str
        Column name used for y-axis.
    raw_df : pd.DataFrame
        Filtered dataframe used in processing.
    processing_output : dict
        Output of TractionDataProcessor.process().
    window_length : float
        Frame duration in seconds.
    label_value : int
        Which frame label to plot.
    pre_peak_time : float, optional
        Duration in seconds for continuation (or until peak for label=1).
    figsize : tuple
        Figure size.
    """

    # --- Extract processor outputs ---
    frame_start = processing_output["frame_start_indices"]
    frame_end = processing_output["frame_end_indices"]
    frame_labels = processing_output["frame_labels"]
    peak_indices = processing_output.get("peak_indices", np.array([]))

    selected_indices = np.where(frame_labels == label_value)[0]
    if len(selected_indices) == 0:
        print(f"No frames found with label={label_value}.")
        return

    # --- Validate signals ---
    for sig in (x_signal_name, y_signal_name):
        if sig not in raw_df.columns:
            raise ValueError(f"Signal '{sig}' not found in dataframe.")

    x_full = raw_df[x_signal_name].to_numpy()
    y_full = raw_df[y_signal_name].to_numpy()

    # --- Time (only for continuation sizing) ---
    if "time" in raw_df.columns:
        time = pd.to_datetime(raw_df["time"], unit="s")
        time_s = (time - time.iloc[0]).dt.total_seconds().to_numpy()
    else:
        dt = 0.01
        time_s = np.arange(len(x_full)) * dt

    dt = np.mean(np.diff(time_s)) if len(time_s) > 1 else 0.01
    fs = 1.0 / dt

    extend_plot = pre_peak_time is not None and pre_peak_time > 0

    # --- Map faulty frames to peaks ---
    frame_to_peak_map = {}
    if extend_plot and label_value == 1 and len(peak_indices) > 0:
        faulty_frames = np.where(frame_labels == 1)[0]
        for i, fidx in enumerate(faulty_frames):
            if i < len(peak_indices):
                frame_e = frame_end[fidx]
                peak_idx = peak_indices[i]
                expected_peak = frame_e + int(round(pre_peak_time * fs))
                if abs(peak_idx - expected_peak) < max(5, 0.2 * pre_peak_time * fs):
                    frame_to_peak_map[fidx] = peak_idx

    # --- Plot ---
    plt.figure(figsize=figsize)

    for idx in selected_indices:
        s = frame_start[idx]
        e = frame_end[idx]

        if s < 0 or e >= len(x_full) or s > e:
            continue

        # Frame data
        x_frame = x_full[s:e + 1]
        y_frame = y_full[s:e + 1]

        plt.plot(x_frame, y_frame, alpha=0.5, linewidth=1.5)

        # --- Continuation ---
        if extend_plot:
            if label_value == 1 and idx in frame_to_peak_map:
                cont_start = e
                cont_end = min(frame_to_peak_map[idx] + 1, len(x_full))
            else:
                cont_start = e
                cont_samples = int(round(pre_peak_time * fs))
                cont_end = min(cont_start + cont_samples + 1, len(x_full))

            if cont_start < cont_end:
                x_cont = x_full[cont_start:cont_end]
                y_cont = y_full[cont_start:cont_end]

                plt.plot(
                    x_cont,
                    y_cont,
                    linestyle="--",
                    linewidth=1.5,
                    alpha=0.5,
                    color="orange",
                )

                # Mark frame end
                plt.scatter(
                    x_frame[-1],
                    y_frame[-1],
                    color="green",
                    s=20,
                    zorder=3,
                )

    plt.xlabel(x_signal_name)
    plt.ylabel(y_signal_name)

    continuation_suffix = " (with continuation)" if extend_plot else ""
    custom_suffix = f" - {title_suffix}" if title_suffix else ""

    plt.title(
        f"Overlapped frames: '{y_signal_name}' vs '{x_signal_name}' "
        f"(label={label_value}){continuation_suffix}{custom_suffix}"
    )

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_overlapping_frames_signal_vs_signal(
    y_signal_name="ground_tether_force",
    x_signal_name="ground_tether_reelout_speed",
    raw_df=results["filtered_data"],
    processing_output=results,
    window_length=window_length,
    label_value=1,
    pre_peak_time=pre_peak_time,
)


In [ ]:
results["filtered_data"]

In [ ]:
from functions.preprocessing.plots import plot_overlapping_frames_for_signal
plot_overlapping_frames_for_signal(
    signal_name="ground_tether_force",
    raw_df=results["filtered_data"],  # Use filtered data to match frame indices
    processing_output=results,
    window_length=processor.window_length,
    label_value=1,
    pre_peak_time=processor.pre_peak_time,
    threshold=results.get("threshold_on_T", threshold_on_T)  # Use threshold from results if available
)